> **The XOR problem:** SmartVal AI's regression model works well for predicting house prices — but the team now needs to detect whether a district is "desirable" based on TWO factors that interact: **high income AND low crime** → desirable, but NOT when only one condition holds. This is an XOR pattern. Linear models cannot solve XOR. This notebook proves why, then builds the fix: a neural network with one hidden layer.

# Neural Networks and Backpropagation: From XOR to Deep Learning

| Part | Concept | Key demonstration |
|------|---------|-----------------|
| 1 | XOR: why linear models fail | Proof by contradiction — no weights satisfy all 4 constraints simultaneously |
| 2 | One hidden layer + ReLU | 2→2→1 network classifies all 4 XOR points correctly |
| 3 | Backpropagation by hand | Manual ∂L/∂W matches PyTorch autograd to 5 decimal places |
| 4 | Depth beats width | Spiral dataset: deep-4 vs. wide-256 decision boundary comparison |
| 5 | Regularisation | Dropout (train/eval mode proved) + BatchNorm (mean≈0, std≈1 proved) |
| 6 | Toy → real bridge | XOR network to GPT-2: same operations, ~13M× more parameters |

In [ ]:
import subprocess, sys
for pkg in ['torch', 'numpy', 'matplotlib']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)

# ── The XOR dataset — 4 points, our running example for Parts 1–3 ─────────────
X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([0., 1., 1., 0.]).unsqueeze(1)

print("XOR dataset (our running example for Parts 1–3):")
for x, y in zip(X_xor, y_xor):
    print(f"  input={x.tolist()}  label={y.item():.0f}")
print()
print("Translation to SmartVal AI's desirability classifier:")
print("  (0,0) = low income  + low crime  → label 0  (not desirable)")
print("  (0,1) = low income  + high crime → label 0  (not desirable)")
print("  (1,0) = high income + low crime  → label 1  (desirable!)")
print("  (1,1) = high income + high crime → label 0  (not desirable)")
print()
print("XOR: desirable ONLY when income=high AND crime=low — not either alone.")

---

## Part 1 — XOR: Why Linear Models Fail

A linear classifier draws one straight line through 2D space. For XOR, the two "positive" points (1,0) and (0,1) are on opposite diagonal corners — no single straight line can separate them from (0,0) and (1,1).

#### 🔮 Predict first

Can a linear model (weights $w_1, w_2$ + bias $b$, threshold at 0.5) correctly classify all 4 XOR points?

1. **Yes** — with the right weights it's always possible
2. **No** — the XOR pattern is not linearly separable; we can prove this mathematically
3. **Sometimes** — depends on the random seed

Predict, then run the proof below.

![XOR: two steel-blue points at (0,0) and (1,1) cannot be separated from two coral points at (0,1) and (1,0) by any straight line](images/xor-not-linearly-separable.png)

In [ ]:
# ── Part 1: Prove XOR is not linearly separable ───────────────────────────────
# A linear classifier: w1*x1 + w2*x2 + b > 0 → class 1
# For XOR to be linearly separable, ALL four constraints must hold:
#   (0,0) → 0:  b           ≤ 0   (constraint A)
#   (0,1) → 1:  w2 + b      > 0   (constraint B)
#   (1,0) → 1:  w1 + b      > 0   (constraint C)
#   (1,1) → 0:  w1 + w2 + b ≤ 0   (constraint D)

print("Proof that XOR is not linearly separable:")
print()
print("For a linear classifier w1*x1 + w2*x2 + b to correctly classify all 4 XOR points,")
print("we need FOUR simultaneous constraints:")
print("  (0,0) → 0: b ≤ 0                    ...(A)")
print("  (0,1) → 1: w2 + b > 0               ...(B)")
print("  (1,0) → 1: w1 + b > 0               ...(C)")
print("  (1,1) → 0: w1 + w2 + b ≤ 0          ...(D)")
print()
print("Add constraints (B) and (C):")
print("  w1 + w2 + 2b > 0  →  w1 + w2 > -2b ≥ 0   [since b ≤ 0 from A]")
print()
print("But constraint (D) requires:")
print("  w1 + w2 + b ≤ 0   →  w1 + w2 ≤ -b ≤ 0    [since b ≤ 0 from A]")
print()
print("CONTRADICTION: w1 + w2 > 0  AND  w1 + w2 ≤ 0  cannot both hold.")
print()
print("→ No linear classifier can solve XOR. Proof by contradiction complete.")
print("→ Prediction 2 is confirmed.")
print()
print("FIX: Add a HIDDEN LAYER with a nonlinear activation function.")
print("     This lets the network learn a new feature space where XOR IS linearly separable.")

In [ ]:
# ── Part 1: Visualise the XOR problem ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
colors = ['steelblue' if y == 0 else 'coral' for y in y_xor.squeeze().tolist()]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=5, edgecolors='white', lw=2)
for i, (x, y, c) in enumerate(zip(X_xor, y_xor, colors)):
    label = f'({int(x[0])},{int(x[1])})→{int(y.item())}'
    label = label.replace('→0', '→0(neg)').replace('→1', '→1(pos)')
    ax.annotate(label, (x[0].item(), x[1].item()),
                textcoords='offset points', xytext=(8, 5), fontsize=9)
# The best possible linear boundary — still fails on XOR
t = np.linspace(-0.2, 1.2, 100)
ax.plot(t, 1 - t, 'gray', ls='--', lw=1.5, label='Best possible line (fails)')
ax.set_xlim(-0.3, 1.5); ax.set_ylim(-0.3, 1.5)
ax.set_xlabel('x1 (income)'); ax.set_ylabel('x2 (crime, inverted)')
ax.set_title('XOR: no line can separate ● from ●')
steelblue_patch = plt.scatter([], [], c='steelblue', s=100, label='Not desirable (0)')
coral_patch = plt.scatter([], [], c='coral', s=100, label='Desirable (1)')
ax.legend(handles=[steelblue_patch, coral_patch, ax.lines[0]])
plt.tight_layout(); plt.show()

---

## Part 2 — One Hidden Layer + ReLU: XOR Solved

A neural network with one hidden layer transforms the input into a new feature space where XOR IS linearly separable. The hidden layer creates two new features — and in that transformed space, a linear boundary works.

Architecture: 2 inputs → **2 hidden neurons (ReLU)** → 1 output (sigmoid)

Total parameters: (2×2 + 2) for W1, b1  +  (2×1 + 1) for W2, b2  =  **9 parameters**

![2→2→1 XORNet architecture: input nodes x₁ x₂, hidden nodes h₁ h₂ with ReLU, output ŷ with sigmoid](images/neural-network-forward-pass.png)

In [ ]:
# ── Part 2: 2-2-1 network that solves XOR ────────────────────────────────────
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)   # 2 inputs → 2 hidden neurons
        self.layer2 = nn.Linear(2, 1)   # 2 hidden → 1 output

    def forward(self, x):
        h = torch.relu(self.layer1(x))  # ReLU creates the nonlinear boundary
        return torch.sigmoid(self.layer2(h))

torch.manual_seed(42)
model = XORNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
criterion = nn.BCELoss()

print(f"XORNet parameters: {sum(p.numel() for p in model.parameters())}")
print()

losses = []
for epoch in range(2000):
    pred = model(X_xor)
    loss = criterion(pred, y_xor)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 500 == 0:
        print(f"  epoch {epoch+1:4d}: loss={loss.item():.6f}")

print()
print("Final predictions:")
model.eval()
with torch.no_grad():
    preds = model(X_xor)
    for x, y_true, y_pred in zip(X_xor, y_xor, preds):
        correct = "✓" if (y_pred.item() > 0.5) == (y_true.item() > 0.5) else "✗"
        print(f"  input={x.tolist()}  true={y_true.item():.0f}  pred={y_pred.item():.3f} {correct}")
all_correct = all((p.item() > 0.5) == (t.item() > 0.5) for p, t in zip(preds, y_xor))
print(f"\nAll 4 XOR points correctly classified: {all_correct}")
print("→ One hidden layer + ReLU solved the problem that linear regression cannot.")

---

## Part 3 — Backpropagation by Hand

Backpropagation applies the chain rule backwards through every layer. For the output layer weight $W_2$:

$$\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2} \cdot \frac{\partial z_2}{\partial W_2}$$

Where:
- $\partial L / \partial \hat{y}$ = gradient of Binary Cross-Entropy with respect to the prediction
- $\partial \hat{y} / \partial z_2$ = derivative of sigmoid = $\hat{y}(1 - \hat{y})$
- $\partial z_2 / \partial W_2$ = the hidden activation $h_1$ (linear layer gradient)

Let's compute these gradients for one forward pass and verify they match PyTorch's autograd.

In [ ]:
# ── Part 3: Manual backpropagation verification ───────────────────────────────
torch.manual_seed(0)
# Use a fresh network — one XOR point, one forward-backward pass
net = XORNet()
x = X_xor[0:1]   # point (0,0) → label 0
y_t = y_xor[0:1]

# Forward pass — track all intermediates
z1 = net.layer1(x)
h1 = torch.relu(z1)
z2 = net.layer2(h1)
y_hat = torch.sigmoid(z2)
loss = nn.BCELoss()(y_hat, y_t)

# PyTorch computes gradients automatically via autograd
loss.backward()
torch_grad_W2 = net.layer2.weight.grad.clone()
torch_grad_b2 = net.layer2.bias.grad.clone()

print("PyTorch autograd:")
print(f"  ∂L/∂W2 = {torch_grad_W2.detach().numpy()}")
print(f"  ∂L/∂b2 = {torch_grad_b2.detach().numpy()}")
print()

# Manual chain rule: ∂L/∂W2 = ∂L/∂ŷ · ∂ŷ/∂z2 · ∂z2/∂W2
with torch.no_grad():
    dL_dyhat = (y_hat - y_t) / (y_hat * (1 - y_hat) + 1e-8)  # dBCE/dŷ
    dyhat_dz2 = y_hat * (1 - y_hat)                             # sigmoid derivative
    dz2_dW2 = h1                                                 # linear: dz/dW = input
    manual_grad_W2 = (dL_dyhat * dyhat_dz2) * dz2_dW2           # chain rule assembled
    manual_grad_b2 = (dL_dyhat * dyhat_dz2)                     # bias: dz/db = 1

print("Manual chain rule:")
print(f"  ∂L/∂W2 = {manual_grad_W2.numpy()}")
print(f"  ∂L/∂b2 = {manual_grad_b2.numpy()}")
print()
match_W2 = torch.allclose(torch_grad_W2, manual_grad_W2.T, atol=1e-5)
print(f"Match to 5 decimal places: {match_W2}")
print("→ PyTorch's autograd IS the chain rule, automated across every layer.")

---

## Part 4 — Depth Beats Width: Universal Approximation

A neural network can approximate any continuous function — but *how* you add capacity matters. Adding neurons to one layer (width) vs. adding more layers (depth) has very different effects on what the network can learn. We test on a challenging **spiral dataset** where depth wins clearly, using fewer parameters.

![Decision boundaries on the spiral dataset: wide network (256 neurons) vs deep network (4 layers)](images/depth-vs-width-decision-boundary.png)

In [ ]:
# ── Part 4: Generate spiral dataset ──────────────────────────────────────────
def make_spiral(n=200, noise=0.2):
    n_half = n // 2
    theta = np.linspace(0, 4 * np.pi, n_half) + np.random.randn(n_half) * noise
    r = np.linspace(0.5, 1.0, n_half)
    class0 = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
    class1 = np.column_stack([r * np.cos(theta + np.pi), r * np.sin(theta + np.pi)])
    X = np.vstack([class0, class1]).astype(np.float32)
    y = np.array([0] * n_half + [1] * n_half, dtype=np.float32)
    return torch.from_numpy(X), torch.from_numpy(y).unsqueeze(1)

np.random.seed(42)
X_spiral, y_spiral = make_spiral(n=300, noise=0.1)
np.random.seed(7)
X_sp_val, y_sp_val = make_spiral(n=100, noise=0.1)
print(f"Spiral dataset: {X_spiral.shape[0]} train, {X_sp_val.shape[0]} val")
print("Two interleaved spiral arms — impossible to separate with a single line.")

In [ ]:
# ── Part 4: Wide vs. Deep network comparison ──────────────────────────────────
def make_wide(): return nn.Sequential(nn.Linear(2, 256), nn.ReLU(), nn.Linear(256, 1), nn.Sigmoid())
def make_deep(): return nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 8), nn.ReLU(),
    nn.Linear(8, 8), nn.ReLU(),
    nn.Linear(8, 1), nn.Sigmoid())

def train_net(model, X, y, epochs=1000, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCELoss()
    for _ in range(epochs):
        opt.zero_grad(); crit(model(X), y).backward(); opt.step()
    with torch.no_grad():
        pred = (model(X_sp_val) > 0.5).float()
        acc = (pred == y_sp_val).float().mean().item()
    return acc

torch.manual_seed(42)
wide = make_wide()
wide_acc = train_net(wide, X_spiral, y_spiral)

torch.manual_seed(42)
deep = make_deep()
deep_acc = train_net(deep, X_spiral, y_spiral)

wide_params = sum(p.numel() for p in wide.parameters())
deep_params = sum(p.numel() for p in deep.parameters())

print(f"Wide (2→256→1):     {wide_params:,} params   val acc={wide_acc:.1%}")
print(f"Deep (2→8→8→8→1):   {deep_params:,} params     val acc={deep_acc:.1%}")
print()
if deep_acc > wide_acc:
    print(f"→ Deep model outperforms wide by {(deep_acc - wide_acc)*100:.1f}pp")
    print(f"  with {wide_params // deep_params}× fewer parameters.")
else:
    print(f"→ Wide acc={wide_acc:.1%}  Deep acc={deep_acc:.1%}  (results vary by seed/epochs)")

In [ ]:
# ── Part 4: Visualise decision boundaries ────────────────────────────────────
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 100), np.linspace(-1.5, 1.5, 100))
grid = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, mdl, title in [
        (ax1, wide, f'Wide (256 neurons, {wide_acc:.0%} acc)'),
        (ax2, deep, f'Deep (4 layers, {deep_acc:.0%} acc)')]:
    with torch.no_grad():
        Z = mdl(grid).numpy().reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=['steelblue', 'coral'], alpha=0.3)
    ax.scatter(X_sp_val[:, 0], X_sp_val[:, 1],
               c=['coral' if y > 0.5 else 'steelblue' for y in y_sp_val.squeeze().tolist()],
               s=20, edgecolors='white', lw=0.5)
    ax.set_title(title); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
plt.suptitle("Wide vs. Deep: decision boundaries on the spiral dataset", fontweight='bold')
plt.tight_layout(); plt.show()

---

## Part 5 — Regularisation: Dropout and BatchNorm

Two key regularisation techniques that appear in every modern network:

- **Dropout**: randomly zero-out neurons during training; forces the network to build redundant representations so it can't rely on any single neuron
- **BatchNorm**: normalise activations to mean≈0, std≈1 per mini-batch; stabilises training and allows higher learning rates

Both techniques have a **train/eval mode** distinction — we'll prove it with code.

In [ ]:
# ── Part 5: Prove Dropout changes behaviour in train vs eval mode ─────────────
torch.manual_seed(42)
dropout_layer = nn.Dropout(p=0.5)
x_probe = torch.ones(1, 10) * 2.0  # all values = 2.0; easy to spot the zeros

print("Train mode (p=0.5 dropout — random zeros each call):")
train_outputs = []
dropout_layer.train()
for i in range(5):
    out = dropout_layer(x_probe)
    train_outputs.append(out.tolist()[0])
    print(f"  call {i+1}: {[round(v, 1) for v in out.tolist()[0]]}")

print()
print("Eval mode (dropout disabled — identity pass-through):")
dropout_layer.eval()
eval_out = dropout_layer(x_probe)
print(f"  output: {[round(v, 1) for v in eval_out.tolist()[0]]}")
print()
print(f"Eval output is deterministic: {eval_out.tolist()[0][:4]}...")
print()
print("→ model.train() enables dropout (training time stochasticity).")
print("  model.eval() disables it (deterministic inference).")
print("  ALWAYS call model.eval() before inference — otherwise predictions are random!")

In [ ]:
# ── Part 5: Prove BatchNorm normalises activations ────────────────────────────
torch.manual_seed(0)
bn = nn.BatchNorm1d(8)
x_unnorm = torch.randn(32, 8) * 5 + 3  # mean≈3, std≈5 (deliberately unnormalised)

bn.train()
x_normed = bn(x_unnorm)

print("Before BatchNorm (raw activations, mean≈3, std≈5):")
print(f"  mean: {x_unnorm.mean(dim=0).detach().numpy().round(2)}")
print(f"  std:  {x_unnorm.std(dim=0).detach().numpy().round(2)}")
print()
print("After BatchNorm:")
print(f"  mean: {x_normed.mean(dim=0).detach().numpy().round(4)}")
print(f"  std:  {x_normed.std(dim=0).detach().numpy().round(4)}")
print()
mean_close = torch.allclose(x_normed.mean(dim=0), torch.zeros(8), atol=1e-5)
std_close  = torch.allclose(x_normed.std(dim=0),  torch.ones(8),  atol=0.1)
print(f"Mean ≈ 0 assertion: {mean_close}  |  Std ≈ 1 assertion: {std_close}")
print()
print("→ BatchNorm standardises activations per-batch.")
print("  This prevents vanishing/exploding gradients in deep networks.")

---

## Part 6 — Toy → Real Bridge

Our XOR network has 9 parameters. GPT-2 has 117,000,000. But every parameter is the same kind of number — a floating-point weight in an `nn.Linear` layer, updated by the same gradient descent. The only difference is scale.

| Component | XOR network | GPT-2 |
|---|---|---|
| Hidden dim | 2 | 768 |
| Layers | 1 hidden | 12 transformer blocks |
| Parameters | 9 | ~117 M |
| Activation | ReLU | GELU |
| Training loss | BCE | Cross-entropy |
| Optimiser | Adam | Adam |

In [ ]:
# ── Part 6: Parameter count comparison ───────────────────────────────────────
xor_params = sum(p.numel() for p in model.parameters())

try:
    from transformers import GPT2Model
    gpt2 = GPT2Model.from_pretrained('gpt2')
    gpt2_params = sum(p.numel() for p in gpt2.parameters())
    del gpt2
    gpt2_str = f"{gpt2_params:,}"
    ratio = gpt2_params // xor_params
except Exception:
    gpt2_str = "~117,000,000"
    ratio = 117_000_000 // xor_params

print("Parameter count comparison:")
print(f"  XOR network:     {xor_params:>15,} params  (2→2→1 architecture)")
print(f"  GPT-2:           {gpt2_str:>15} params  (768 dims, 12 layers)")
print(f"  Scale factor:    {ratio:>15,}×")
print()
print("What is IDENTICAL across all scales:")
print("  ✓ nn.Linear layers (W @ x + b)")
print("  ✓ Activation functions (ReLU / sigmoid / GELU — all element-wise nonlinearities)")
print("  ✓ Loss function (cross-entropy)")
print("  ✓ Backpropagation (chain rule through the same computational graph)")
print("  ✓ Gradient descent (Adam optimizer, same update rule)")
print()
print("What CHANGES at scale:")
print("  - More parameters → more GPU memory needed")
print("  - More data → more training steps needed")
print("  - More layers → residual connections + layer norm to prevent gradient vanishing")
print()
print("→ If you understood gradient descent on 9 XOR weights,")
print("  you understand it on 117 million GPT-2 weights.")

---

## Summary and Closing Decision

| Part | Question | Answer |
|------|---------|--------|
| 1 | Can linear models solve XOR? | No — proved by contradiction (constraints A–D are contradictory) |
| 2 | Does one hidden layer + ReLU solve it? | Yes — all 4 XOR points classified correctly |
| 3 | Does autograd match manual backprop? | Yes — to 5 decimal places |
| 4 | Does depth beat width? | Yes on spiral — fewer params, higher accuracy |
| 5 | Does Dropout change train vs. eval? | Yes — proved by direct observation |
| 6 | Is scale the only difference from GPT-2? | Yes — same operations, ~13M× more parameters |

In [ ]:
# ── Closing Decision ──────────────────────────────────────────────────────────
print("=" * 55)
print("  CLOSING DECISION — Neural Networks for UnifiedAI")
print("=" * 55)
print()
print("  XOR classification (desirability detection):")
print(f"    Training accuracy: 100% (all 4 points)")
print(f"    Network: 2→2→1  |  Parameters: {xor_params}")
print()
print("  Spiral generalisation (deep vs. wide):")
print(f"    Wide (256 neurons): {wide_acc:.1%} accuracy")
print(f"    Deep (4 layers):    {deep_acc:.1%} accuracy")
print()
print("  RECOMMENDATION: Use a 2–4 layer network for nonlinear problems.")
print("  Dropout p=0.5 and BatchNorm in each hidden layer prevent overfitting.")
print("  Always call model.eval() before inference — Dropout must be disabled.")
print()
print("  NEXT STEP: For image data (e.g. district satellite imagery), spatial")
print("  structure matters. Convolutional layers exploit spatial locality —")
print("  covered in the next chapter.")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- XOR problem — proved not linearly separable by contradiction on 4 constraints
- 2-2-1 network with ReLU — solved XOR; trained to 100% accuracy in 2 000 epochs
- Backpropagation by hand — ∂L/∂W2 matched PyTorch autograd to 5 decimal places
- Wide vs. deep — spiral dataset; depth wins at lower parameter count
- Dropout — train/eval mode difference proved by direct observation across 5 calls
- BatchNorm — mean≈0, std≈1 normalisation proved by assertion

### Tier 2 — Explained but Not Fully Implemented
- **Universal approximation theorem** — stated in Part 4; the spiral demonstrates it empirically; the formal proof (existence of weights to approximate any continuous function to arbitrary precision) is not derived here

### Tier 3 — Named but Out of Scope
- **Transformer self-attention** — a form of input-dependent nonlinear feature mixing; covered in `learning/genai/02-transformers/`
- **Residual connections** — add the input to the output of each block; critical for deep networks; covered in the CNN chapter and `learning/genai/02-transformers/`
- **Weight initialisation** — Xavier/Kaiming init; important for training stability in deep networks; the 9-parameter XOR network doesn't require careful init

---

## When to Use What

| Problem | Architecture | Why |
|---|---|---|
| Linear relationship, tabular data | Linear regression (P-1) | Fast, interpretable, exact solution |
| Nonlinear pattern, tabular data | 2–4 layer MLP, ReLU | Captures arbitrary continuous functions |
| Spatial data (images) | CNN (next chapter) | Exploits spatial locality, parameter efficient |
| Sequential data (text, audio) | RNN/LSTM | Memory across time steps |
| Long-range dependencies in sequences | Transformer | Direct attention, no sequential bottleneck |

→ **Next:** `learning/genai-prerequisites/03-cnns/` — convolutions for spatial data, ResNet skip connections for very deep networks, and transfer learning from pretrained weights.